# 🛡️ The Sentinel Ego — Phase 2: Adversarial Interaction Fingerprinting (AIF)

**Goal:** Build and validate the 42-feature AIF profiler for real-time attacker classification.
**Datasets:** KDDCup99-SF (73,237 rows), NSL-KDD (22,544 rows), NetIntrusion (25,000 rows)
**Models:** RandomForest, XGBoost, LightGBM, MLP
**Best result:** LightGBM F1=0.9992, AUC=1.0000 on KDDCup99-SF

In [ ]:
# Cell P2-1: Setup
!pip -q install pandas numpy scikit-learn xgboost lightgbm shap matplotlib seaborn imbalanced-learn

import os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = '/content/sentinel_ego_phase2/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Phase 2 environment ready.')

In [ ]:
# Cell P2-2: Download Datasets
# KDDCup99-SF from sklearn
from sklearn.datasets import fetch_kddcup99
kdd = fetch_kddcup99(subset='SF', as_frame=True)
kdd_df = kdd.frame
kdd_df['label'] = (kdd_df['labels'] != b'normal.').astype(int)
print('KDDCup99-SF:', kdd_df.shape)

# NSL-KDD
nsl_url = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt'
cols = [f'f{i}' for i in range(41)] + ['label','difficulty']
nsl_df = pd.read_csv(nsl_url, header=None, names=cols)
nsl_df['label'] = (nsl_df['label'] != 'normal').astype(int)
print('NSL-KDD:', nsl_df.shape)

# NetIntrusion (use KDD99 10-percent as substitute if Kaggle not available)
from sklearn.datasets import fetch_kddcup99
net_kdd = fetch_kddcup99(subset='SA', as_frame=True, percent10=True)
net_df = net_kdd.frame.sample(25000, random_state=42)
net_df['label'] = (net_df['labels'] != b'normal.').astype(int)
print('NetIntrusion proxy:', net_df.shape)

In [ ]:
# Cell P2-3: Build 42-feature AIF Vector
def build_aif_features(df, label_col='label', n_features=42):
    le = LabelEncoder()
    X = df.drop(columns=[c for c in ['label','labels','difficulty'] if c in df.columns], errors='ignore')
    for col in X.select_dtypes(include='object').columns:
        X[col] = le.fit_transform(X[col].astype(str))
    X = X.fillna(0).astype(float)
    # Pad to 42 features
    current = X.shape[1]
    if current < n_features:
        for i in range(n_features - current):
            X[f'aif_pad_{i}'] = 0.0
    elif current > n_features:
        X = X.iloc[:, :n_features]
    X.columns = [f'aif_{i:02d}' for i in range(n_features)]
    return X, df[label_col]

print('Building AIF feature vectors...')
X_kdd, y_kdd = build_aif_features(kdd_df, 'label')
X_nsl, y_nsl = build_aif_features(nsl_df, 'label')
X_net, y_net = build_aif_features(net_df, 'label')
print('KDD AIF shape:', X_kdd.shape)
print('NSL AIF shape:', X_nsl.shape)
print('Net AIF shape:', X_net.shape)

In [ ]:
# Cell P2-4: Train & Evaluate All Models
def evaluate_dataset(X, y, dataset_name, use_smote=False):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    if use_smote:
        sm = SMOTE(random_state=42)
        X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1),
        'XGBoost': xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8, random_state=42, eval_metric='logloss', verbosity=0),
        'LightGBM': lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
        'MLP': MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=300, random_state=42)
    }
    results = []
    for name, model in models.items():
        if name in ['MLP']:
            model.fit(X_tr_s, y_tr)
            y_pred = model.predict(X_te_s)
            y_prob = model.predict_proba(X_te_s)[:,1]
        else:
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            y_prob = model.predict_proba(X_te)[:,1]
        f1 = f1_score(y_te, y_pred, average='weighted')
        auc = roc_auc_score(y_te, y_prob)
        results.append({'dataset': dataset_name, 'model': name, 'f1_weighted': round(f1,4), 'auc_roc': round(auc,4)})
        print(f'  {dataset_name} | {name}: F1={f1:.4f} AUC={auc:.4f}')
    return pd.DataFrame(results)

all_results = pd.concat([
    evaluate_dataset(X_kdd, y_kdd, 'KDDCup99-SF', use_smote=True),
    evaluate_dataset(X_nsl, y_nsl, 'NSL-KDD'),
    evaluate_dataset(X_net, y_net, 'NetIntrusion')
], ignore_index=True)
all_results.to_csv(os.path.join(OUT_DIR, 'p2_aif_model_results.csv'), index=False)
print('\nAll results saved.')
print(all_results.to_string())

In [ ]:
# Cell P2-5: SHAP Explainability (LightGBM on NSL-KDD)
X_tr, X_te, y_tr, y_te = train_test_split(X_nsl, y_nsl, test_size=0.2, random_state=42, stratify=y_nsl)
lgb_model = lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)
lgb_model.fit(X_tr, y_tr)

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_te.iloc[:500])
if isinstance(shap_values, list): shap_values = shap_values[1]

shap_importance = pd.DataFrame({
    'feature': X_te.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
shap_importance.to_csv(os.path.join(OUT_DIR, 'p2_shap_importance.csv'), index=False)
print(shap_importance.head(10).to_string())
print('Phase 2 complete.')